Accumulate results

In [1]:
import pandas as pd

base_path = "stackPredAMR_Results"

antibiotics = ["Amikacin_AB", "Gentamicin_AB", "Piperacillin-tazobactam_AB"]

combinations = [
    ("mix", "na"),
    ("mix", "phoenix"),
    ("mix", "vitek"),
    ("phoenix", "na"),
    ("phoenix", "phoenix"),
    ("phoenix", "vitek"),
]

results = []

for training, test in combinations:

    for fold in range(5):

        folder = f"Evaluation_CV_Conclusio_" f"{training}Train_{test}Test_fold{fold}"

        file_path = f"{base_path}/{folder}/Evaluation.csv"

        df = pd.read_csv(file_path, index_col=0)

        for antibiotic in antibiotics:
            results.append(
                {
                    "Training": training,
                    "Test": test,
                    "Fold": fold,
                    "Antibiotic": antibiotic,
                    "Accuracy": df.loc[antibiotic, "Accuracy"],
                }
            )

results_df = pd.DataFrame(results)
results_df.to_csv("statistics/Accuracy_combined.csv", index=False)
print(results_df)

   Training   Test  Fold                  Antibiotic  Accuracy
0       mix     na     0                 Amikacin_AB  0.857143
1       mix     na     0               Gentamicin_AB  0.891429
2       mix     na     0  Piperacillin-tazobactam_AB  0.717391
3       mix     na     1                 Amikacin_AB  0.807692
4       mix     na     1               Gentamicin_AB  0.834356
..      ...    ...   ...                         ...       ...
85  phoenix  vitek     3               Gentamicin_AB  0.880000
86  phoenix  vitek     3  Piperacillin-tazobactam_AB  0.784232
87  phoenix  vitek     4                 Amikacin_AB  0.786667
88  phoenix  vitek     4               Gentamicin_AB  0.870000
89  phoenix  vitek     4  Piperacillin-tazobactam_AB  0.808000

[90 rows x 5 columns]


In [4]:
import pandas as pd

base_path = "stackPredAMR_Results"

antibiotics = ["Amikacin_AB", "Gentamicin_AB", "Piperacillin-tazobactam_AB"]

combinations = [
    ("mix", "na"),
    ("mix", "phoenix"),
    ("mix", "vitek"),
    ("phoenix", "na"),
    ("phoenix", "phoenix"),
    ("phoenix", "vitek"),
]

results = []

for training, test in combinations:

    for fold in range(5):

        folder = f"Evaluation_CV_Conclusio_" f"{training}Train_{test}Test_fold{fold}"

        file_path = f"{base_path}/{folder}/Evaluation.csv"

        df = pd.read_csv(file_path, index_col=0)

        for antibiotic in antibiotics:
            results.append(
                {
                    "Training": training,
                    "Test": test,
                    "Fold": fold,
                    "Antibiotic": antibiotic,
                    "ROC_AUC": df.loc[antibiotic, "ROC_Mean"],
                }
            )

results_df = pd.DataFrame(results)
results_df.to_csv("statistics/ROC_combined.csv", index=False)
print(results_df)

   Training   Test  Fold                  Antibiotic   ROC_AUC
0       mix     na     0                 Amikacin_AB  0.940698
1       mix     na     0               Gentamicin_AB  0.951422
2       mix     na     0  Piperacillin-tazobactam_AB  0.804584
3       mix     na     1                 Amikacin_AB  0.896869
4       mix     na     1               Gentamicin_AB  0.938788
..      ...    ...   ...                         ...       ...
85  phoenix  vitek     3               Gentamicin_AB  0.923414
86  phoenix  vitek     3  Piperacillin-tazobactam_AB  0.842827
87  phoenix  vitek     4                 Amikacin_AB  0.791505
88  phoenix  vitek     4               Gentamicin_AB  0.908423
89  phoenix  vitek     4  Piperacillin-tazobactam_AB  0.872585

[90 rows x 5 columns]


In [8]:
import pandas as pd

base_path = "stackPredAMR_Results"

antibiotics = ["Amikacin_AB", "Gentamicin_AB", "Piperacillin-tazobactam_AB"]

combinations = [
    ("mix", "na"),
    ("mix", "phoenix"),
    ("mix", "vitek"),
    ("phoenix", "na"),
    ("phoenix", "phoenix"),
    ("phoenix", "vitek"),
]

results = []

for training, test in combinations:

    for fold in range(5):

        folder = f"Evaluation_CV_Conclusio_" f"{training}Train_{test}Test_fold{fold}"

        file_path = f"{base_path}/{folder}/Evaluation.csv"

        df = pd.read_csv(file_path, index_col=0)

        for antibiotic in antibiotics:
            results.append(
                {
                    "Training": training,
                    "Test": test,
                    "Fold": fold,
                    "Antibiotic": antibiotic,
                    "F1-Score": df.loc[antibiotic, "F1_Mean"],
                }
            )

results_df = pd.DataFrame(results)
results_df.to_csv("statistics/F1_combined.csv", index=False)
print(results_df)

   Training   Test  Fold                  Antibiotic  F1-Score
0       mix     na     0                 Amikacin_AB  0.832891
1       mix     na     0               Gentamicin_AB  0.889992
2       mix     na     0  Piperacillin-tazobactam_AB  0.683765
3       mix     na     1                 Amikacin_AB  0.781513
4       mix     na     1               Gentamicin_AB  0.834256
..      ...    ...   ...                         ...       ...
85  phoenix  vitek     3               Gentamicin_AB  0.879808
86  phoenix  vitek     3  Piperacillin-tazobactam_AB  0.756755
87  phoenix  vitek     4                 Amikacin_AB  0.658022
88  phoenix  vitek     4               Gentamicin_AB  0.869964
89  phoenix  vitek     4  Piperacillin-tazobactam_AB  0.786096

[90 rows x 5 columns]


Two sided (mixed vs phoenix) paired mutation test for accuracy

In [12]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

# Map each input file to its metric column.
METRIC_COLUMNS = {
    "Accuracy_combined.csv": "Accuracy",
    "ROC_combined.csv": "ROC_AUC",
    "F1_combined.csv": "F1-Score",
}


def analyze_mix_vs_phoenix(
    df_input,
    n_bootstrap=10000,
    alternative="two-sided",
    random_state=42,
    *,
    metric="Accuracy",
):
    keys = ["Test", "Fold", "Antibiotic"]
    required_columns = {"Training", *keys, metric}
    missing = required_columns - set(df_input.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if alternative not in {"two-sided", "greater"}:
        raise ValueError("alternative must be 'two-sided' or 'greater'.")
    if n_bootstrap < 1:
        raise ValueError("n_bootstrap must be positive.")

    data = df_input.loc[df_input["Training"].isin(["mix", "phoenix"])].copy()
    if data.empty:
        raise ValueError("No rows found for either 'mix' or 'phoenix'.")
    if data[keys].isna().any().any():
        raise ValueError("Missing values in Test, Fold, or Antibiotic.")
    if data.duplicated(["Training", *keys]).any():
        raise ValueError(
            "Duplicate combination of Training, Test, Fold, and Antibiotic."
        )
    data[metric] = pd.to_numeric(data[metric], errors="raise")
    if not np.isfinite(data[metric]).all():
        raise ValueError(f"Column {metric} contains missing or infinite values.")

    paired = data.pivot(index=keys, columns="Training", values=metric)
    paired = paired.reindex(columns=["mix", "phoenix"])
    incomplete = paired[paired.isna().any(axis=1)]
    if not incomplete.empty:
        raise ValueError(f"Incomplete mix/phoenix pairs for {metric}:\n{incomplete}")

    paired["difference"] = paired["mix"] - paired["phoenix"]
    blocks = paired.groupby(level=["Test", "Fold"], observed=True).median()
    differences = blocks["difference"].to_numpy()
    n_blocks = len(differences)
    observed_difference = differences.mean()

    # Enumerate all sign flips across test and fold blocks.
    n_permutations = 2**n_blocks
    n_extreme = 0
    for signs in product((-1, 1), repeat=n_blocks):
        statistic = np.mean(differences * signs)
        if alternative == "greater":
            n_extreme += int(statistic >= observed_difference)
        else:
            n_extreme += int(abs(statistic) >= abs(observed_difference))
    p_value = n_extreme / n_permutations

    rng = np.random.default_rng(random_state)
    bootstrap_differences = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        sample = rng.choice(differences, size=n_blocks, replace=True)
        bootstrap_differences[i] = sample.mean()
    ci_lower, ci_upper = np.percentile(bootstrap_differences, [2.5, 97.5])

    return {
        "n_blocks": n_blocks,
        "n_test_datasets": data["Test"].nunique(),
        "n_folds": data["Fold"].nunique(),
        "n_antibiotics": data["Antibiotic"].nunique(),
        f"mean_mix_{metric.lower()}": blocks["mix"].mean(),
        f"mean_phoenix_{metric.lower()}": blocks["phoenix"].mean(),
        "mean_difference": observed_difference,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "p_value": p_value,
        "alternative": alternative,
        "significant_0.05": p_value < 0.05,
        "n_permutations": n_permutations,
    }


def main():
    directory = Path("statistics")
    results = {}
    for filename, metric in METRIC_COLUMNS.items():
        result = analyze_mix_vs_phoenix(
            pd.read_csv(directory / filename), metric=metric
        )
        # Use the same row labels for all three metrics.
        result["mean_mix"] = result.pop(f"mean_mix_{metric.lower()}")
        result["mean_phoenix"] = result.pop(f"mean_phoenix_{metric.lower()}")
        results[metric] = result

    table = pd.DataFrame(results).rename_axis("Metric")
    table.to_csv(directory / "results.csv")
    print(table.to_string())


if __name__ == "__main__":
    main()

                   Accuracy    ROC_AUC   F1-Score
Metric                                           
n_blocks                 15         15         15
n_test_datasets           3          3          3
n_folds                   5          5          5
n_antibiotics             3          3          3
mean_difference    0.052561   0.038347   0.053377
ci_lower           0.030007   0.018236   0.027734
ci_upper           0.075673   0.058372   0.079847
p_value            0.000671   0.003357   0.001648
alternative       two-sided  two-sided  two-sided
significant_0.05       True       True       True
n_permutations        32768      32768      32768
mean_mix           0.870774   0.934533   0.849178
mean_phoenix       0.812991    0.88637   0.794077
